# House Price Prediction: preprocessing and baseline

Builds the cleaning pipeline from the decisions made during EDA (see `03_eda.ipynb`) and scores a Linear Regression baseline with 5-fold cross-validation. The test set is not used here.

## 1. Load and split

Split into train and test with a fixed `random_state`. Houses 523 and 1298 are removed from the training set only (unfinished `Partial` sales whose prices don't match their size and quality).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression


df = pd.read_csv("data/ames_housing.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.drop(index=[523, 1298])


## 2. Features and target

`SalePrice` is the target. `Id` is dropped because it carries no information. `GarageYrBlt` and `PoolArea` are dropped (see the cleaning decisions in `03_eda.ipynb`).

In [2]:
X_train = train_df.drop(columns=['SalePrice', 'Id', 'GarageYrBlt', 'PoolArea'])
y_train = train_df['SalePrice']

X_test = test_df.drop(columns=['SalePrice', 'Id', 'GarageYrBlt', 'PoolArea'])
y_test = test_df['SalePrice']

In [3]:
print(f"Shape of X_train: {X_train.shape}; Shape of y_train: {y_train.shape}")
print(f"Shape of X_test: {X_test.shape}; Shape of y_test: {y_test.shape}")


Shape of X_train: (1166, 77); Shape of y_train: (1166,)
Shape of X_test: (292, 77); Shape of y_test: (292,)


## 3. Numeric and categorical columns

`MSSubClass` is a house-type code stored as a number, so it is moved to the categorical list.

In [4]:
numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
categorical_cols = X_train.select_dtypes(exclude='number').columns.tolist()


numeric_cols.remove('MSSubClass')
categorical_cols.append('MSSubClass')

In [5]:
print(f"The length of Numerical columns:{len(numeric_cols)}\n The length of Categorical columns: {len(categorical_cols)}")

The length of Numerical columns:33
 The length of Categorical columns: 44


## 4. Preprocessing pipeline

All of this is fitted on the training set only.

### Numeric branch
Fill missing values with the median, then standardize.

In [6]:
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('scaler', StandardScaler())])

X_num = numeric_pipeline.fit_transform(X_train[numeric_cols])
                            

In [7]:
print(X_train[numeric_cols].isna().sum().sum())
print(pd.isna(X_num).sum())

223
0


### Categorical branch
A blank means the feature doesn't exist (no garage, no pool), so it is filled with "None". Then one-hot encode: categories with fewer than 20 houses are grouped together, and categories unseen in training are ignored instead of crashing.

In [8]:
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),
                                 ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False))])

X_cat = categorical_pipeline.fit_transform(X_train[categorical_cols])

                                


In [9]:
print(X_train[categorical_cols].isna().sum().sum())
print(pd.isna(X_cat).sum())
print(X_cat.shape)


5933
0
(1166, 207)


### Combined
`ColumnTransformer` applies each branch to its own columns and joins the results.

In [10]:
preprocessor = ColumnTransformer([('num', numeric_pipeline, numeric_cols),
                                  ('cat', categorical_pipeline, categorical_cols)])

X_prepared = preprocessor.fit_transform(X_train) 


In [11]:
print(X_prepared.shape) 
print(np.isnan(X_prepared).sum())

(1166, 240)
0


## 5. Baseline model

Preprocessor and Linear Regression in one pipeline, scored with 5-fold cross-validation on the training set. Every later model has to beat these numbers.

In [12]:
baseline = Pipeline([('prep', preprocessor),
                     ('model', LinearRegression())])

scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Mean of MAE:{-scores.mean()}")

scores_rmse = cross_val_score(baseline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
print(f"Mean of RMSE:{-scores_rmse.mean()}")

scores_r2 = cross_val_score(baseline, X_train, y_train, cv=5, scoring='r2')
print(f"Mean of R2 score:{scores_r2.mean()}")

print(f"R2 scores:{scores_r2}")

Mean of MAE:17207.463617697722
Mean of RMSE:25395.88625441448
Mean of R2 score:0.8896753334786069
R2 scores:[0.90616818 0.8910277  0.89663503 0.89993563 0.85461013]
